In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070 Ti


In [ ]:
import json
import random

random.seed(42) # 12, 42, 72
path = r'C:\rf-detr\aquarium\train\_annotations.coco.json'

with open(path, 'r') as file:
    coco = json.load(file)

# Store images based on class

images_by_cls = {}

for anno in coco['annotations']:
    if anno['category_id'] not in images_by_cls:
        images_by_cls[anno['category_id']] = set()
    images_by_cls[anno['category_id']].add(anno['image_id'])

few_shot = set()
for cls, images in images_by_cls.items():
    n_min = min(25, len(images))
    random_select = random.sample(images, 25)
    few_shot.update(list(random_select))

images_coco = [img for img in coco['images'] if img['id'] in few_shot]
annotations_coco = [anno for anno in coco['annotations'] if anno['image_id'] in few_shot]

new_coco = {
    'categories': coco['categories'],
    'images': images_coco,
    'annotations': annotations_coco
}

saved_path = path = r'C:\rf-detr\vertebra\train\_annotations_trial.coco.json'

with open(saved_path, 'w') as files:
    json.dump(new_coco, files, indent= 4)

since Python 3.9 and will be removed in a subsequent version.


## Convert Coco to YOLO

In [4]:
import json
import os
import shutil

path = r'C:\rf-detr\vertebra\train\_annotations.coco.json'
orig = r'C:\ultralytics-main\examples\datasets\scoliosis\train\images'
yolo_path = r'C:\ultralytics-main\examples\datasets\scoliosis\train_25'

with open(path, "r") as f:
    coco = json.load(f)

# build image_id → info
id_to_img = {img["id"]: img for img in coco["images"]}

os.makedirs(yolo_path + '\labels', exist_ok=True)
os.makedirs(yolo_path + "\images", exist_ok=True)

# Map to avoid copying the same image multiple times
copied_images = set()

for ann in coco["annotations"]:
    img = id_to_img[ann["image_id"]]
    img_name = img["file_name"]
    text_name = img_name[:-4]

    # Copy the image only once
    if img_name not in copied_images:
        shutil.copy(os.path.join(orig, img_name), os.path.join(yolo_path + "\images", img_name))
        copied_images.add(img_name)

    # YOLO format expects class IDs starting from 0
    class_id = ann['category_id']  # adjust if your COCO starts from 1

    w, h = img["width"], img["height"]
    x, y, bw, bh = ann["bbox"]

    xc = (x + bw / 2) / w
    yc = (y + bh / 2) / h
    bw /= w
    bh /= h
    
    label_path = f'{yolo_path}/labels/{text_name}.txt'
    with open(label_path, "a") as f:
        f.write(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

## Add superclass for COCO annotations before tranining

In [3]:
from myfile import add_superclass

all_status = ['train']
super_name = "creatures"

for all in all_status:
    path = rf'C:\rf-detr\aquarium_data_25\{all}'
    add_superclass(path, super_name)

## Remove irrevelant labels and remapping correct indices

In [1]:
from myfile import remove_label

all_status = ['valid', 'test']

dataset = 'aquarium_data_25'

for all in all_status:
    remove_label(all, dataset)

since Python 3.9 and will be removed in a subsequent version.


In [3]:
import os, json

path = r'C:\rf-detr\floor_plans\valid'
label_path = os.path.join(path, '_annotations.coco.json')

new_img = []
all_img = {f for f in os.listdir(path) if f.endswith('.jpg')}  # collect once

with open(label_path, 'r') as file:
    coco = json.load(file)

# Filter images
for img in coco['images']:
    if img['file_name'] in all_img:
        new_img.append(img)

# Optionally, filter annotations too
image_ids = {img['id'] for img in new_img}
new_annotations = [ann for ann in coco['annotations'] if ann['image_id'] in image_ids]

# Save filtered JSON
filtered_coco = coco.copy()
filtered_coco['images'] = new_img
filtered_coco['annotations'] = new_annotations

with open(os.path.join(path, '_annotations.coco.json'), 'w') as f:
    json.dump(filtered_coco, f, indent=2)

print(f"Filtered JSON saved, {len(new_img)} images and {len(new_annotations)} annotations")


Filtered JSON saved, 6 images and 113 annotations


In [6]:
import json
import os
from PIL import Image

def resize_img(path, label_path, save_path, ml=1024):
    with open(label_path, "r") as f:
        coco = json.load(f)

    for img_info in coco["images"]:
        img_name = img_info["file_name"]
        img_id = img_info["id"]

        img_path = os.path.join(path, img_name)
        img = Image.open(img_path)

        old_w, old_h = img.size
        max_len = max(old_w, old_h)

        if max_len <= ml:
            img.save(os.path.join(path, img_name))
            continue

        scale = ml / max_len
        new_w = int(old_w * scale)
        new_h = int(old_h * scale)

        resized_img = img.resize((new_w, new_h), Image.LANCZOS)
        resized_img.save(os.path.join(path, img_name))

        # update image size
        img_info["width"] = new_w
        img_info["height"] = new_h

        # update only its bboxes
        for anno in coco["annotations"]:
            if anno["image_id"] == img_id:
                x, y, w, h = anno["bbox"]
                anno["bbox"] = [
                    x * scale,
                    y * scale,
                    w * scale,
                    h * scale
                ]

    with open(os.path.join(path, "_annotations.coco.json"), "w") as f:
        json.dump(coco, f, indent=4)

        
cls = ['train'] 
for c in cls:      
    path = rf'C:\rf-detr\fire_data_25\{c}'
    label_path = os.path.join(path, '_annotations.coco.json')
    save_path = os.path.join(path, '_annotations.coco.json')
    with open(label_path, 'r') as file:
        coco = json.load(file)

    max_h, max_w = 0, 0
    min_h, min_w = 0, 0

    for img in coco['images']:
        max_h = max(max_h, img['height'])
        max_w = max(max_w, img['width'])

        min_h = max(min_h, img['height'])
        min_w = max(min_w, img['width'])

    print(max_h, max_w)
    print(min_h, min_w)

    if max_h > 1024 or max_w > 1024:
        resize_img(path, label_path, save_path)



1280 1920
1280 1920


In [1]:
from rfdetr import RFDETRNano, RFDETRMedium
import torch

torch.cuda.empty_cache()

In [2]:
from rfdetr import RFDETRNano,RFDETRLarge

model = RFDETRLarge()
cls = 'vertebra'
dataset_path = rf'C:\rf-detr\{cls}'

model.train(
    dataset_dir=dataset_path, 
    epochs=100, 
    resolution=640,
    batch_size=4 ,
    warmup_epochs = 5,
    two_stage = True,
    lr_scheduler = 'cosine',
    grad_accum_steps=4,
    use_autocast=False,
    early_stopping = True,
    early_stopping_patience = 20,
    early_stopping_min_delta = 0.0005,
    num_workers=0,
    lr=1e-4,              # Recommended starting learning rate
    output_dir=rf"C:\rf-detr\output\{cls}" # Path to save checkpoints
)
#model.train(dataset_dir=dataset.location, epochs=10, batch_size=8, grad_accum_steps=2)

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights
Unable to initialize TensorBoard. Logging is turned off for this session.  Run 'pip install tensorboard' to enable logging.
Not using distributed mode
git:
  sha: fd1295b8ccacba0fad2b4a40c8a35b67bc62c335, status: has uncommited changes, branch: develop

Namespace(num_classes=1, grad_accum_steps=4, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=4, weight_decay=0.0001, epochs=100, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='rf-detr-large-2026.p

Epoch: [0]  [0/5]  eta: 0:00:16  lr: 0.000010  class_error: 0.00  loss: 24.0089 (24.0089)  loss_ce: 0.2036 (0.2036)  loss_bbox: 1.7887 (1.7887)  loss_giou: 2.2472 (2.2472)  loss_ce_0: 0.1364 (0.1364)  loss_bbox_0: 2.8000 (2.8000)  loss_giou_0: 2.4129 (2.4129)  loss_ce_1: 0.1745 (0.1745)  loss_bbox_1: 2.2113 (2.2113)  loss_giou_1: 2.2669 (2.2669)  loss_ce_2: 0.1872 (0.1872)  loss_bbox_2: 1.9762 (1.9762)  loss_giou_2: 2.2528 (2.2528)  loss_ce_enc: 0.1516 (0.1516)  loss_bbox_enc: 2.8255 (2.8255)  loss_giou_enc: 2.3741 (2.3741)  loss_ce_unscaled: 0.2036 (0.2036)  class_error_unscaled: 0.0000 (0.0000)  loss_bbox_unscaled: 0.3577 (0.3577)  loss_giou_unscaled: 1.1236 (1.1236)  cardinality_error_unscaled: 14.2500 (14.2500)  loss_ce_0_unscaled: 0.1364 (0.1364)  loss_bbox_0_unscaled: 0.5600 (0.5600)  loss_giou_0_unscaled: 1.2064 (1.2064)  cardinality_error_0_unscaled: 14.2500 (14.2500)  loss_ce_1_unscaled: 0.1745 (0.1745)  loss_bbox_1_unscaled: 0.4423 (0.4423)  loss_giou_1_unscaled: 1.1334 (1.13

## Validation

In [3]:
import supervision as sv
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision, Precision, Recall
from supervision.metrics import AveragingMethod
from rfdetr import RFDETRLarge
import numpy as np
from pycocotools.coco import COCO

#cls = 'aquarium_data'


model = RFDETRLarge(pretrain_weights=rf'C:\rf-detr\output\{cls}\checkpoint_best_total.pth')
model.optimize_for_inference()

annotations_path=rf'C:\rf-detr\{cls}/valid/_annotations.coco.json'

ds = sv.DetectionDataset.from_coco(
    images_directory_path=rf'C:\rf-detr\{cls}/valid',
    annotations_path=annotations_path
)

coco = COCO(annotations_path)
coco.loadCats(coco.getCatIds())
cats = coco.loadCats(coco.getCatIds())
id2name = {cat['id']: cat['name'] for cat in cats}

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path).convert("RGB")
    
    detections = model.predict(image, threshold=0.5)

    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

precision_metric = Precision(averaging_method=AveragingMethod.MACRO, )
recall_metric = Recall(averaging_method=AveragingMethod.MACRO, )

precision_result = precision_metric.update(predictions, targets).compute()
recall_result = recall_metric.update(predictions, targets).compute()

precision_list = [precision[0] for precision in precision_result.precision_per_class]
recall_list = [recall[0] for recall in recall_result.recall_per_class]

print('\n')
print(f"{'Class':<25} {'Precision':<10} {'Recall':<10} {'mAP@50':<10} {'mAP@50-95':<10}")
print(f"{'All':<25} {precision_result.precision_at_50:<10.3f} {recall_result.recall_at_50:<10.3f} {map_result.map50:<10.3f} {map_result.map50_95:<10.3f}")

for i, (p, r, ap) in enumerate(zip(precision_list, recall_list, map_result.ap_per_class), start=1):

    p_val = 0.0 if np.isnan(p) else p
    r_val = 0.0 if np.isnan(r) else r
    print(f"{id2name[i]:<25} {p_val:<10.3f} {r_val:<10.3f} {ap[0]:<10.3f} {np.mean(ap):<10.3f}")



Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|██████████| 100/100 [00:04<00:00, 21.91it/s]




Class                     Precision  Recall     mAP@50     mAP@50-95 
All                       0.879      0.884      0.857      0.423     


KeyError: 1

## Testing

In [4]:
import supervision as sv
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision, Precision, Recall
from supervision.metrics import AveragingMethod
from rfdetr import RFDETRLarge
import numpy as np
from pycocotools.coco import COCO

#cls = 'entrance_data'


model = RFDETRLarge(pretrain_weights=rf'C:\rf-detr\output\{cls}\checkpoint_best_total.pth')
model.optimize_for_inference()

annotations_path=rf'C:\rf-detr\{cls}/test/_annotations.coco.json'

ds = sv.DetectionDataset.from_coco(
    images_directory_path=rf'C:\rf-detr\{cls}/test',
    annotations_path=annotations_path
)

coco = COCO(annotations_path)
coco.loadCats(coco.getCatIds())
cats = coco.loadCats(coco.getCatIds())
id2name = {cat['id']: cat['name'] for cat in cats}

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path).convert("RGB")
    
    detections = model.predict(image, threshold=0.5)

    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

precision_metric = Precision(averaging_method=AveragingMethod.MACRO, )
recall_metric = Recall(averaging_method=AveragingMethod.MACRO, )

precision_result = precision_metric.update(predictions, targets).compute()
recall_result = recall_metric.update(predictions, targets).compute()

precision_list = [precision[0] for precision in precision_result.precision_per_class]
recall_list = [recall[0] for recall in recall_result.recall_per_class]

print('\n')
print(f"{'Class':<25} {'Precision':<10} {'Recall':<10} {'mAP@50':<10} {'mAP@50-95':<10}")
print(f"{'All':<25} {precision_result.precision_at_50:<10.3f} {recall_result.recall_at_50:<10.3f} {map_result.map50:<10.3f} {map_result.map50_95:<10.3f}")

for i, (p, r, ap) in enumerate(zip(precision_list, recall_list, map_result.ap_per_class), start=1):

    p_val = 0.0 if np.isnan(p) else p
    r_val = 0.0 if np.isnan(r) else r
    print(f"{id2name[i]:<25} {p_val:<10.3f} {r_val:<10.3f} {ap[0]:<10.3f} {np.mean(ap):<10.3f}")



Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|██████████| 100/100 [00:04<00:00, 24.00it/s]




Class                     Precision  Recall     mAP@50     mAP@50-95 
All                       0.903      0.854      0.835      0.439     


KeyError: 1

## Prediction

In [ ]:
import supervision as sv
from PIL import Image
import glob, os
from pycocotools.coco import COCO

cls = 'entrance_data'

path = rf'C:\rf-detr\{cls}\test'
saved_path = r"C:\rf-detr\entrance_data_pred_image"

model = RFDETRLarge(pretrain_weights=rf'C:\rf-detr\output\{cls}\checkpoint_best_total.pth')

annotation_file = rf'C:\rf-detr\{cls}\train\_annotations.coco.json'
anno_file = COCO(annotation_file)
cat_ids = anno_file.getCatIds()
cats = anno_file.loadCats(cat_ids)
class_names = [c['name'] for c in cats]

for image_path in glob.glob(path + '/' + '*.jpg'):
    file_name = os.path.basename(image_path)
    image = Image.open(image_path)
    detections = model.predict(image, threshold=0.2)

    labels = [f"{class_names[class_id]} {confidence:.2f}" for class_id, confidence in zip(detections.class_id, detections.confidence)]

    annotated_image = sv.BoxAnnotator().annotate(image, detections)
    annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

    save_dir = os.path.join(saved_path, file_name)

    annotated_image.save(save_dir)
print("Successfully saved!")

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Successfully saved!


## Video

In [16]:
import cv2
import torch
from PIL import Image
import supervision as sv
import os
import glob
from rfdetr import RFDETRLarge
import numpy as np
import time

cls = "fire_data1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = RFDETRLarge(
    pretrain_weights=rf"C:\rf-detr\output\{cls}\checkpoint_best_total.pth"
)

model.optimize_for_inference()

video_folder = r"C:\ultralytics-main\examples\datasets\videos\fire"
save_folder = r"C:\rf-detr\videos"

os.makedirs(save_folder, exist_ok=True)

videos = glob.glob(os.path.join(video_folder, "*.mp4"))

class_names = {1: "fire", 2: "smoke"}

latencies = []
frame_count = 0

for video_path in videos:

    video_name = os.path.basename(video_path)
    save_path = os.path.join(save_folder, f"rf_{video_name}")

    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(save_path, fourcc, fps, (w, h))

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        image = Image.fromarray(
            cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        )

        # -------- Measure inference latency --------
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        detections = model.predict(image, threshold=0.2)
        
        if device == "cuda":
            torch.cuda.synchronize()

        end = time.time()

        latencies.append(end - start)
        # -------------------------------------------

        labels = [
            f"{class_names[class_id]} {confidence:.2f}"
            for class_id, confidence in zip(
                detections.class_id, detections.confidence
            )
        ]

        annotated_image = sv.BoxAnnotator().annotate(image, detections)
        annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)
        annotated_cv = cv2.cvtColor(np.array(annotated_image), cv2.COLOR_RGB2BGR)

        out.write(annotated_cv)

    cap.release()
    out.release()

# -------- Latency statistics --------
avg_latency = sum(latencies) / len(latencies)
avg_latency_ms = avg_latency * 1000

fps_from_latency = 1 / avg_latency

print(f"Total Frames: {frame_count}")
print(f"Average Latency: {avg_latency_ms:.5f} ms")
print(f"Inference FPS: {fps_from_latency:.5f}")

Using device: cuda
Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights
Total Frames: 6943
Average Latency: 60.19883 ms
Inference FPS: 16.61162
